In [35]:
import scipy.io
import networkx as nx 
import bct 
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd

# Preparation of dataset for use
number_subjects = 9

# definition of directory with dataset
dir_fmri_desikan = '/strombolihome/fribeiro/Dataset/source_reconstructed_FC/fmri_connect_desikan/'
dir_fmri_destrieux = '/strombolihome/fribeiro/Dataset/source_reconstructed_FC/fmri_connect_destrieux/'

dir_eeg_desikan = '/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/'
dir_eeg_destrieux = '/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_destrieux/'

In [5]:
# Script for creating graphs from fMRI and EEG connectivity data (coverting MATLAB matrices)

#1.Creation of average Graph 
def createAvGraph(file,data_type):
    
    # Load Matlab file with connectivity matrix for each time point - 3D matrix
    mat = scipy.io.loadmat(file)

    if data_type == 'fmri':
        conn_matrix = mat['connFMRI'].transpose() #connectivity fMRI matrix with numpy format
    elif data_type == 'eeg': #for now only for broad band TODO others !
        conn_matrix = mat['connEEGbroad'].transpose() #connectivity EEG matrix with numpy format
    elif data_type == 'eeg_alpha':
        conn_matrix = mat['connEEGalpha'].transpose()
    elif data_type == 'eeg_beta':
        conn_matrix = mat['connEEGbeta'].transpose()
    elif data_type == 'eeg_delta':
        conn_matrix = mat['connEEGdelta'].transpose()
    elif data_type == 'eeg_gamma':
        conn_matrix = mat['connEEGgamma'].transpose()
    elif data_type == 'eeg_theta':
        conn_matrix = mat['connEEGtheta'].transpose()
        
    t_points = conn_matrix.shape[0] # number of layers of multilayer matrix
    num_areas = conn_matrix.shape[1] #number of nodes of the graph
    
    # Get average connectivity matrix
    av_conn = np.zeros((num_areas,num_areas))
       
    for i in range(t_points):
        
        av_conn = av_conn + conn_matrix[i]
            
    av_conn = av_conn/t_points 
    
    G = nx.from_numpy_matrix(av_conn)
    
    return G

#2. Creation of array of graphs (equivalent to layers)
def createArrayGraph(file,data_type):
    
    # Load Matlab file with connectivity matrix for each time point - 3D matrix
    mat = scipy.io.loadmat(file)
    
    if data_type == 'fmri':
        conn_matrix = mat['connFMRI'].transpose() #connectivity fMRI matrix with numpy format
    elif data_type == 'eeg': #for now only for broad band TODO others !
        conn_matrix = mat['connEEGbroad'].transpose() #connectivity EEG matrix with numpy format
    elif data_type == 'eeg_alpha':
        conn_matrix = mat['connEEGalpha'].transpose()
    elif data_type == 'eeg_beta':
        conn_matrix = mat['connEEGbeta'].transpose()
    elif data_type == 'eeg_delta':
        conn_matrix = mat['connEEGdelta'].transpose()
    elif data_type == 'eeg_gamma':
        conn_matrix = mat['connEEGgamma'].transpose()
    elif data_type == 'eeg_theta':
        conn_matrix = mat['connEEGtheta'].transpose()

    t_points = conn_matrix.shape[0] # number of layers of multilayer matrix
    num_areas = conn_matrix.shape[1] #number of nodes of the graph
    
    array_graphs = np.empty(t_points, dtype=object) 
    
    for i in range(t_points):
        array_graphs[i] = nx.from_numpy_matrix(conn_matrix[i])
        
    return array_graphs
    

In [21]:
#3.Find minimum threshold that keeps giant component (hold at least 90% of the graph's nodes) - fMRI data

# Function to obtain the components of a graph plotting the giant component and the second biggest component
def getComponents(G):
    
    graph_components = sorted(nx.connected_components(G), key=len, reverse=True)
        
    giant = G.subgraph(graph_components[0]).copy() #giant component for this threshold
    #nx.draw_networkx(giant)
    #plt.figure(figsize=(100, 100))
    #plt.show()
    print("Number of nodes of the giant component:", giant.number_of_nodes())
        
    if len(graph_components) > 1 :
        second = G.subgraph(graph_components[1]).copy()
        #nx.draw_networkx(second)
        #plt.figure(figsize=(50, 50))
        #plt.show()
        print("Number of nodes of the second biggest component:", second.number_of_nodes())
        
    return graph_components
              
    
# Function to compare giant component and the second biggest component, varying the threshold used
def findThresholdGiantComponent(G, proportion_values):
    
    # get absolute value connectivity matrix
    conn_matrix = abs(nx.to_numpy_array(G))
     
    # to obtain connectivity matrix thresholded and the corresponding "giant" component
    for p in proportion_values:
        
        print("Proportion kept:", p*100)
        new = bct.utils.threshold_proportional(conn_matrix, p, True) # option to use a proportional threshold - BCT
        G=nx.from_numpy_matrix(new)
        nodes = G.number_of_nodes()
        edges = G.number_of_edges()
        print("Number of edges of thresholded graph:", edges)
        av_degree = (2*edges)/nodes
        print("Average degree of thresholded graph: ", av_degree)
        
        graph_components = getComponents(G)
        
        print("___________________________________________________________________________")
        
    # return value only matters to compare giant component    
    if len(graph_components) > 1:
        return (graph_components[0].number_of_nodes() - graph_components[1].number_of_nodes())
    else:
        return 0
    


In [7]:
# 4.Check giant component for each time point of each subject - fMRI data and EEG data

def checkGiantComponent(G_array, proportion_values, atlas):
    
    array_problem_time_points = np.zeros(len(G_array))
    t = 0
    
    #arbitrary difference defined between giant and second biggest component
    if atlas == 'dsk':
        limit = 15 
    elif atlas == 'dstrx':
        limit = 30
    
    for G in G_array:
        t +=  1
        print("Graph for time point", t)
        
        difference = findThresholdGiantComponent(G,proportion_values)
            
        if difference < limit and difference != 0:
                array_problem_time_points[t-1] = t
           
    return array_problem_time_points

In [20]:
#5. Check size distribution of graph components for each time point - fMRI and EEG data

def distComponents(G_array,threshold,data):
    
    time_point = 0
    
    for G in G_array:
        time_point += 1
        print("Components for time frame: ", time_point)
        
        if(data == 'fmri'):
            # get absolute value connectivity matrix
            conn_matrix = abs(nx.to_numpy_array(G))
            # to threshold graph keeping top X% of the edges
            new = bct.utils.threshold_proportional(conn_matrix, threshold, True) 
            G=nx.from_numpy_matrix(new)
        
        graph_components = sorted(nx.connected_components(G), key=len, reverse=True)
        size_components = np.zeros(len(graph_components))
        
        for i in range(0,len(graph_components)):
            nodes = graph_components[i].number_of_nodes()
            size_components[i] = nodes
       
        print("Number of components:", len(graph_components))
        print("Size of each component:", size_components)
        
        # plotting the number of nodes, i.e, size of each component
        plt.bar(np.arange(1,len(graph_components)+1),size_components, width = 0.7, align='center', tick_label = np.arange(1,len(graph_components)+1), alpha=0.5)
        plt.ylabel("Number of nodes")
        plt.xlabel("Components")
        plt.title("Distribution of the components' size")

        plt.show() 
        
        #create save of these results !!! TODO TODO TODO TODO TODO
    

In [13]:
#6. Component and threshold analysis functions for each subject, data type and atlas

def thresholdAnalysis(subject, type_data, atlas, file, threshold_values):
    
    print("Subject {}, with {} and using {}".format(subject,type_data,atlas))
    
    #print(file)
    
    G = createAvGraph(file, type_data)
    
    difference = findThresholdGiantComponent(G, threshold_values[atlas])

# to be run for a given threshold (in array shape)
def componentAnalysis(subject, type_data, atlas, file, threshold):
    
    G_array = createArrayGraph(file, type_data)
    
    problem_time_points = checkGiantComponent(G_array, threshold, atlas)
    print("These are the time frames for which the giant component was compromised, for subject {}, with {} and using {} atlas: {}".format(subject,type_data,atlas,problem_time_points))
    
    distComponents(G_array, threshold, type_data)

In [11]:
# proportion of top edges to keep of the graph
proportion = {'dsk': [0.01, 0.015, 0.016, 0.018, 0.02, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.25, 0.5, 1, 1, 1, 1, 1], 'dstrx': [0.005, 0.008, 0.01, 0.012, 0.015, 0.018, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.25, 0.5, 1]}

threshold_values = pd.DataFrame(data = proportion)

print(threshold_values)


      dsk  dstrx
0   0.010  0.005
1   0.015  0.008
2   0.016  0.010
3   0.018  0.012
4   0.020  0.015
5   0.050  0.018
6   0.060  0.020
7   0.070  0.030
8   0.080  0.040
9   0.090  0.050
10  0.100  0.060
11  0.250  0.070
12  0.500  0.080
13  1.000  0.090
14  1.000  0.100
15  1.000  0.250
16  1.000  0.500
17  1.000  1.000


In [22]:
# Desikan atlas - fMRI

s = 1

for subdir, dirs, files in sorted(os.walk(dir_fmri_desikan)):
    
    for file in files:
        
        if 'conn_desi_phase_coh' in (os.path.join(subdir,file)):
        
            thresholdAnalysis(s,'fmri','dsk',os.path.join(subdir, file), threshold_values)
            s += 1


Threshold analysis for subject 1, with fmri and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/fmri_connect_desikan/subj01-7T/conn_desi_phase_coh_time_fmri.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 6
Number of nodes of the second biggest component: 6
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 9
Number of nodes of the second biggest component: 8
___________________________________________________________________________
Proportion kept: 1.6
Number of edges of thresholded graph: 36
Average degree of thresholded graph:  1.0588235294117647
Number of nodes of the giant component: 9
Number of nodes of the second biggest component: 9
________________________________________________

Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 3, with fmri and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/fmri_connect_desikan/subj03-7T/conn_desi_phase_coh_time_fmri.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 11
Number of nodes of the second biggest component: 4
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph: 

Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 5, with fmri and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/fmri_connect_desikan/subj05-7T/conn_desi_phase_coh_time_fmri.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 7
Number of nodes of the second biggest component: 5
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 8
Number of nodes of the second biggest co

Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 7, with fmri and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/fmri_connect_desikan/subj07-7T/conn_desi_phase_coh_time_fmri.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 7
Number of nodes of the second biggest component: 6
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 19
Number of nodes of the second biggest c

Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 9, with fmri and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/fmri_connect_desikan/subj09-7T/conn_desi_phase_coh_time_fmri.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 9
Number of nodes of the second biggest component: 3
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 13
Number of nodes of the second biggest c

In [ ]:
# Desikan atlas - fMRI

componentAnalysis(1,'fmri','dsk', dir_fmri_desikan + '/sub01-7T/conn_desi_phase_coh_time_fmri.mat', [0.1])
#componentAnalysis(2,'fmri','dsk', dir_fmri_desikan + '/sub02-7T/conn_desi_phase_coh_time_fmri.mat', [0.1])
#componentAnalysis(3,'fmri','dsk', dir_fmri_desikan + '/sub03-7T/conn_desi_phase_coh_time_fmri.mat', [0.1])
#componentAnalysis(4,'fmri','dsk', dir_fmri_desikan + '/sub04-7T/conn_desi_phase_coh_time_fmri.mat', [0.1])
#componentAnalysis(5,'fmri','dsk', dir_fmri_desikan + '/sub05-7T/conn_desi_phase_coh_time_fmri.mat', [0.1])
#componentAnalysis(6,'fmri','dsk', dir_fmri_desikan + '/sub06-7T/conn_desi_phase_coh_time_fmri.mat', [0.1])
#componentAnalysis(7,'fmri','dsk', dir_fmri_desikan + '/sub07-7T/conn_desi_phase_coh_time_fmri.mat', [0.1])
#componentAnalysis(8,'fmri','dsk', dir_fmri_desikan + '/sub08-7T/conn_desi_phase_coh_time_fmri.mat', [0.1])
#componentAnalysis(9,'fmri','dsk', dir_fmri_desikan + '/sub09-7T/conn_desi_phase_coh_time_fmri.mat', [0.1])

In [23]:
# Destrieux atlas - fMRI

s = 1

for subdir, dirs, files in sorted(os.walk(dir_fmri_destrieux)):
    
    for file in files:
        
        if 'conn_destrx_phase_coh' in (os.path.join(subdir,file)):
    
            thresholdAnalysis(s,'fmri','dstrx',os.path.join(subdir, file), threshold_values)
            s += 1
        

Threshold analysis for subject 1, with fmri and using dstrx
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/fmri_connect_destrieux/subj01-7T/conn_destrx_phase_coh_time_fmri.mat
Proportion kept: 0.5
Number of edges of thresholded graph: 54
Average degree of thresholded graph:  0.7297297297297297
Number of nodes of the giant component: 16
Number of nodes of the second biggest component: 8
___________________________________________________________________________
Proportion kept: 0.8
Number of edges of thresholded graph: 87
Average degree of thresholded graph:  1.1756756756756757
Number of nodes of the giant component: 31
Number of nodes of the second biggest component: 9
___________________________________________________________________________
Proportion kept: 1.0
Number of edges of thresholded graph: 109
Average degree of thresholded graph:  1.472972972972973
Number of nodes of the giant component: 54
Number of nodes of the second biggest component: 9
________________________

Number of edges of thresholded graph: 2720
Average degree of thresholded graph:  36.75675675675676
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
Threshold analysis for subject 3, with fmri and using dstrx
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/fmri_connect_destrieux/subj03-7T/conn_destrx_phase_coh_time_fmri.mat
Proportion kept: 0.5
Number of edges of thresholded graph: 54
Average degree of thresholded graph:  0.7297297297297297
Number of nodes of

Number of edges of thresholded graph: 2720
Average degree of thresholded graph:  36.75675675675676
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
Threshold analysis for subject 5, with fmri and using dstrx
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/fmri_connect_destrieux/subj05-7T/conn_destrx_phase_coh_time_fmri.mat
Proportion kept: 0.5
Number of edges of thresholded graph: 54
Average degree of thresholded graph:  0.7297297297297297
Number of nodes of

Number of edges of thresholded graph: 2720
Average degree of thresholded graph:  36.75675675675676
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
Threshold analysis for subject 7, with fmri and using dstrx
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/fmri_connect_destrieux/subj07-7T/conn_destrx_phase_coh_time_fmri.mat
Proportion kept: 0.5
Number of edges of thresholded graph: 54
Average degree of thresholded graph:  0.7297297297297297
Number of nodes of

Number of edges of thresholded graph: 2720
Average degree of thresholded graph:  36.75675675675676
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
Threshold analysis for subject 9, with fmri and using dstrx
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/fmri_connect_destrieux/subj09-7T/conn_destrx_phase_coh_time_fmri.mat
Proportion kept: 0.5
Number of edges of thresholded graph: 54
Average degree of thresholded graph:  0.7297297297297297
Number of nodes of

In [ ]:
# Destrieux atlas - fMRI

componentAnalysis(1,'fmri','dstrx', dir_fmri_destrieux + '/sub01-7T/conn_destrx_phase_coh_time_fmri.mat', [0.06])
#componentAnalysis(2,'fmri','dstrx', dir_fmri_destrieux + '/sub02-7T/conn_destrx_phase_coh_time_fmri.mat', [0.06])
#componentAnalysis(3,'fmri','dstrx', dir_fmri_destrieux + '/sub03-7T/conn_destrx_phase_coh_time_fmri.mat', [0.06])
#componentAnalysis(4,'fmri','dstrx', dir_fmri_destrieux + '/sub04-7T/conn_destrx_phase_coh_time_fmri.mat', [0.06])
#componentAnalysis(5,'fmri','dstrx', dir_fmri_destrieux + '/sub05-7T/conn_destrx_phase_coh_time_fmri.mat', [0.06])
#componentAnalysis(6,'fmri','dstrx', dir_fmri_destrieux + '/sub06-7T/conn_destrx_phase_coh_time_fmri.mat', [0.06])
#componentAnalysis(7,'fmri','dstrx', dir_fmri_destrieux + '/sub07-7T/conn_destrx_phase_coh_time_fmri.mat', [0.06])
#componentAnalysis(8,'fmri','dstrx', dir_fmri_destrieux + '/sub08-7T/conn_destrx_phase_coh_time_fmri.mat', [0.06])
#componentAnalysis(9,'fmri','dstrx', dir_fmri_destrieux + '/sub09-7T/conn_destrx_phase_coh_time_fmri.mat', [0.06])

In [27]:
# Desikan atlas - EEG broad band

s = 1

for subdir, dirs, files in sorted(os.walk(dir_eeg_desikan)):
    
    for file in files:
        
        if 'conn_desi_cohi_time_eeg_broad' in (os.path.join(subdir,file)):
        
            thresholdAnalysis(s,'eeg','dsk',os.path.join(subdir, file), threshold_values)
            s += 1


Threshold analysis for subject 1, with eeg and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/subj01-7T/conn_desi_cohi_time_eeg_broad_subj1-7T_.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 7
Number of nodes of the second biggest component: 4
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 12
Number of nodes of the second biggest component: 6
___________________________________________________________________________
Proportion kept: 1.6
Number of edges of thresholded graph: 36
Average degree of thresholded graph:  1.0588235294117647
Number of nodes of the giant component: 12
Number of nodes of the second biggest component: 6
______________________________________

Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 3, with eeg and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/subj03-7T/conn_desi_cohi_time_eeg_broad_subj3-7T_.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 11
Number of nodes of the second biggest component: 4
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 14
Number of nodes of the second 

Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 5, with eeg and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/subj05-7T/conn_desi_cohi_time_eeg_broad_subj5-7T_.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 6
Number of nodes of the second biggest component: 5
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 9
Number of nodes of the second bi

Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 7, with eeg and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/subj07-7T/conn_desi_cohi_time_eeg_broad_subj7-7T_.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 11
Number of nodes of the second biggest component: 5
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 12
Number of nodes of the second 

Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 9, with eeg and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/subj09-7T/conn_desi_cohi_time_eeg_broad_subj9-7T_.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 5
Number of nodes of the second biggest component: 3
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 9
Number of nodes of the second bi

In [ ]:
# Desikan atlas - EEG broad band - TODO DEFINE THRESHOLD!

componentAnalysis(1,'eeg','dsk', dir_eeg_desikan + '/sub01-7T/conn_desi_cohi_time_eeg_broad_subj1-7T_.mat', )
#componentAnalysis(2,'eeg','dsk', dir_eeg_desikan + '/sub02-7T/conn_desi_cohi_time_eeg_broad_subj2-7T_.mat', )
#componentAnalysis(3,'eeg','dsk', dir_eeg_desikan + '/sub03-7T/conn_desi_cohi_time_eeg_broad_subj3-7T_.mat', )
#componentAnalysis(4,'eeg','dsk', dir_eeg_desikan + '/sub04-7T/conn_desi_cohi_time_eeg_broad_subj4-7T_.mat', )
#componentAnalysis(5,'eeg','dsk', dir_eeg_desikan + '/sub05-7T/conn_desi_cohi_time_eeg_broad_subj5-7T_.mat', )
#componentAnalysis(6,'eeg','dsk', dir_eeg_desikan + '/sub06-7T/conn_desi_cohi_time_eeg_broad_subj6-7T_.mat', )
#componentAnalysis(7,'eeg','dsk', dir_eeg_desikan + '/sub07-7T/conn_desi_cohi_time_eeg_broad_subj7-7T_.mat', )
#componentAnalysis(8,'eeg','dsk', dir_eeg_desikan + '/sub08-7T/conn_desi_cohi_time_eeg_broad_subj8-7T_.mat', )
#componentAnalysis(9,'eeg','dsk', dir_eeg_desikan + '/sub09-7T/conn_desi_cohi_time_eeg_broad_subj9-7T_.mat', )


In [28]:
# Desikan atlas - EEG alpha band

s = 1

for subdir, dirs, files in sorted(os.walk(dir_eeg_desikan)):
    
    for file in files:
        
        if 'conn_desi_cohi_time_eeg_alpha' in (os.path.join(subdir,file)):
        
            thresholdAnalysis(s,'eeg_alpha','dsk',os.path.join(subdir, file), threshold_values)
            s += 1


Threshold analysis for subject 1, with eeg_alpha and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/subj01-7T/conn_desi_cohi_time_eeg_alpha_subj1-7T_.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 11
Number of nodes of the second biggest component: 5
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 12
Number of nodes of the second biggest component: 6
___________________________________________________________________________
Proportion kept: 1.6
Number of edges of thresholded graph: 36
Average degree of thresholded graph:  1.0588235294117647
Number of nodes of the giant component: 12
Number of nodes of the second biggest component: 6
_______________________________

Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 3, with eeg_alpha and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/subj03-7T/conn_desi_cohi_time_eeg_alpha_subj3-7T_.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 10
Number of nodes of the second biggest component: 5
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thres

Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 5, with eeg_alpha and using dsk
/strombolihome/fribeiro/Dataset/source_reconstru

Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 7, with eeg_alpha and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/subj07-7T/conn_desi_cohi_time_eeg_alpha_subj7-7T_.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 10
Number of nodes of the second biggest component: 3
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 13
Number of nodes of the s

Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 9, with eeg_alpha and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/subj09-7T/conn_desi_cohi_time_eeg_alpha_subj9-7T_.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 6
Number of nodes of the second biggest component: 4
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 9
Number of nodes of the sec

In [ ]:
# Desikan atlas - EEG alpha band - TODO DEFINE THRESHOLD!

componentAnalysis(1,'eeg','dsk', dir_eeg_desikan + '/sub01-7T/conn_desi_cohi_time_eeg_alpha_subj1-7T_.mat', treshold_values)
#componentAnalysis(2,'eeg','dsk', dir_eeg_desikan + '/sub02-7T/conn_desi_cohi_time_eeg_alpha_subj2-7T_.mat', treshold_values)
#componentAnalysis(3,'eeg','dsk', dir_eeg_desikan + '/sub03-7T/conn_desi_cohi_time_eeg_alpha_subj3-7T_.mat', treshold_values)
#componentAnalysis(4,'eeg','dsk', dir_eeg_desikan + '/sub04-7T/conn_desi_cohi_time_eeg_alpha_subj4-7T_.mat', treshold_values)
#componentAnalysis(5,'eeg','dsk', dir_eeg_desikan + '/sub05-7T/conn_desi_cohi_time_eeg_alpha_subj5-7T_.mat', treshold_values)
#componentAnalysis(6,'eeg','dsk', dir_eeg_desikan + '/sub06-7T/conn_desi_cohi_time_eeg_alpha_subj6-7T_.mat', treshold_values)
#componentAnalysis(7,'eeg','dsk', dir_eeg_desikan + '/sub07-7T/conn_desi_cohi_time_eeg_alpha_subj7-7T_.mat', treshold_values)
#componentAnalysis(8,'eeg','dsk', dir_eeg_desikan + '/sub08-7T/conn_desi_cohi_time_eeg_alpha_subj8-7T_.mat', treshold_values)
#componentAnalysis(9,'eeg','dsk', dir_eeg_desikan + '/sub09-7T/conn_desi_cohi_time_eeg_alpha_subj9-7T_.mat', treshold_values)

In [29]:
# Desikan atlas - EEG beta band

s = 1

for subdir, dirs, files in sorted(os.walk(dir_eeg_desikan)):
    
    for file in files:
        
        if 'conn_desi_cohi_time_eeg_beta' in (os.path.join(subdir,file)):
        
            thresholdAnalysis(s,'eeg_beta','dsk',os.path.join(subdir, file), threshold_values)
            s += 1


Threshold analysis for subject 1, with eeg_beta and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/subj01-7T/conn_desi_cohi_time_eeg_beta_subj1-7T_.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 7
Number of nodes of the second biggest component: 5
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 8
Number of nodes of the second biggest component: 7
___________________________________________________________________________
Proportion kept: 1.6
Number of edges of thresholded graph: 36
Average degree of thresholded graph:  1.0588235294117647
Number of nodes of the giant component: 12
Number of nodes of the second biggest component: 7
___________________________________

Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 3, with eeg_beta and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/subj03-7T/conn_desi_cohi_time_eeg_beta_subj3-7T_.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 10
Number of nodes of the second biggest component: 3
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 13
Number of nodes of the sec

Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 5, with eeg_beta and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/subj05-7T/conn_desi_cohi_time_eeg_beta_subj5-7T_.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 11
Number of nodes of the second biggest component: 5
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 21
Number of nodes of the sec

Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 7, with eeg_beta and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/subj07-7T/conn_desi_cohi_time_eeg_beta_subj7-7T_.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 6
Number of nodes of the second biggest component: 3
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 9
Number of nodes of the secon

Number of nodes of the giant component: 68
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 2278
Average degree of thresholded graph:  67.0
Number of nodes of the giant component: 68
___________________________________________________________________________
Threshold analysis for subject 9, with eeg_beta and using dsk
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/subj09-7T/conn_desi_cohi_time_eeg_beta_subj9-7T_.mat
Proportion kept: 1.0
Number of edges of thresholded graph: 23
Average degree of thresholded graph:  0.6764705882352942
Number of nodes of the giant component: 5
Number of nodes of the second biggest component: 5
___________________________________________________________________________
Proportion kept: 1.5
Number of edges of thresholded graph: 34
Average degree of thresholded graph:  1.0
Number of nodes of the giant component: 18
Number of nodes of the seco

In [ ]:
# Desikan atlas - EEG beta band

componentAnalysis(1,'eeg','dsk', dir_eeg_desikan + '/sub01-7T/conn_desi_cohi_time_eeg_beta_subj1-7T_.mat', treshold_values)
#componentAnalysis(2,'eeg','dsk', dir_eeg_desikan + '/sub02-7T/conn_desi_cohi_time_eeg_beta_subj2-7T_.mat', treshold_values)
#componentAnalysis(3,'eeg','dsk', dir_eeg_desikan + '/sub03-7T/conn_desi_cohi_time_eeg_beta_subj3-7T_.mat', treshold_values)
#componentAnalysis(4,'eeg','dsk', dir_eeg_desikan + '/sub04-7T/conn_desi_cohi_time_eeg_beta_subj4-7T_.mat', treshold_values)
#componentAnalysis(5,'eeg','dsk', dir_eeg_desikan + '/sub05-7T/conn_desi_cohi_time_eeg_beta_subj5-7T_.mat', treshold_values)
#componentAnalysis(6,'eeg','dsk', dir_eeg_desikan + '/sub06-7T/conn_desi_cohi_time_eeg_beta_subj6-7T_.mat', treshold_values)
#componentAnalysis(7,'eeg','dsk', dir_eeg_desikan + '/sub07-7T/conn_desi_cohi_time_eeg_beta_subj7-7T_.mat', treshold_values)
#componentAnalysis(8,'eeg','dsk', dir_eeg_desikan + '/sub08-7T/conn_desi_cohi_time_eeg_beta_subj8-7T_.mat', treshold_values)
#componentAnalysis(9,'eeg','dsk', dir_eeg_desikan + '/sub09-7T/conn_desi_cohi_time_eeg_beta_subj9-7T_.mat', treshold_values)

In [ ]:
# Desikan atlas - EEG delta band

s = 1

for subdir, dirs, files in sorted(os.walk(dir_eeg_desikan)):
    
    for file in files:
        
        if 'conn_desi_cohi_time_eeg_delta' in (os.path.join(subdir,file)):
        
            thresholdAnalysis(s,'eeg_delta','dsk',os.path.join(subdir, file), threshold_values)
            s += 1


In [ ]:
# Desikan atlas - EEG delta band

componentAnalysis(1,'eeg','dsk', dir_eeg_desikan + '/sub01-7T/conn_desi_cohi_time_eeg_delta_subj1-7T_.mat', treshold_values)
#componentAnalysis(2,'eeg','dsk', dir_eeg_desikan + '/sub02-7T/conn_desi_cohi_time_eeg_delta_subj2-7T_.mat', treshold_values)
#componentAnalysis(3,'eeg','dsk', dir_eeg_desikan + '/sub03-7T/conn_desi_cohi_time_eeg_delta_subj3-7T_.mat', treshold_values)
#componentAnalysis(4,'eeg','dsk', dir_eeg_desikan + '/sub04-7T/conn_desi_cohi_time_eeg_delta_subj4-7T_.mat', treshold_values)
#componentAnalysis(5,'eeg','dsk', dir_eeg_desikan + '/sub05-7T/conn_desi_cohi_time_eeg_delta_subj5-7T_.mat', treshold_values)
#componentAnalysis(6,'eeg','dsk', dir_eeg_desikan + '/sub06-7T/conn_desi_cohi_time_eeg_delta_subj6-7T_.mat', treshold_values)
#componentAnalysis(7,'eeg','dsk', dir_eeg_desikan + '/sub07-7T/conn_desi_cohi_time_eeg_delta_subj7-7T_.mat', treshold_values)
#componentAnalysis(8,'eeg','dsk', dir_eeg_desikan + '/sub08-7T/conn_desi_cohi_time_eeg_delta_subj8-7T_.mat', treshold_values)
#componentAnalysis(9,'eeg','dsk', dir_eeg_desikan + '/sub09-7T/conn_desi_cohi_time_eeg_delta_subj9-7T_.mat', treshold_values)

In [3]:
# Desikan atlas - EEG gamma band

s = 1

for subdir, dirs, files in sorted(os.walk(dir_eeg_desikan)):
    
    for file in files:
        
        if 'conn_desi_cohi_time_eeg_gamma' in (os.path.join(subdir,file)):
        
            thresholdAnalysis(s,'eeg_gamma','dsk',os.path.join(subdir, file), threshold_values)
            s += 1


NameError: name 'thresholdAnalysis' is not defined

In [ ]:
# Desikan atlas - EEG gamma band

componentAnalysis(1,'eeg','dsk', dir_eeg_desikan + '/sub01-7T/conn_desi_cohi_time_eeg_gamma_subj1-7T_.mat', treshold_values)
#componentAnalysis(2,'eeg','dsk', dir_eeg_desikan + '/sub02-7T/conn_desi_cohi_time_eeg_gamma_subj2-7T_.mat', treshold_values)
#componentAnalysis(3,'eeg','dsk', dir_eeg_desikan + '/sub03-7T/conn_desi_cohi_time_eeg_gamma_subj3-7T_.mat', treshold_values)
#componentAnalysis(4,'eeg','dsk', dir_eeg_desikan + '/sub04-7T/conn_desi_cohi_time_eeg_gamma_subj4-7T_.mat', treshold_values)
#componentAnalysis(5,'eeg','dsk', dir_eeg_desikan + '/sub05-7T/conn_desi_cohi_time_eeg_gamma_subj5-7T_.mat', treshold_values)
#componentAnalysis(6,'eeg','dsk', dir_eeg_desikan + '/sub06-7T/conn_desi_cohi_time_eeg_gamma_subj6-7T_.mat', treshold_values)
#componentAnalysis(7,'eeg','dsk', dir_eeg_desikan + '/sub07-7T/conn_desi_cohi_time_eeg_gamma_subj7-7T_.mat', treshold_values)
#componentAnalysis(8,'eeg','dsk', dir_eeg_desikan + '/sub08-7T/conn_desi_cohi_time_eeg_gamma_subj8-7T_.mat', treshold_values)
#componentAnalysis(9,'eeg','dsk', dir_eeg_desikan + '/sub09-7T/conn_desi_cohi_time_eeg_gamma_subj9-7T_.mat', treshold_values)

In [ ]:
# Desikan atlas - EEG theta band

s = 1

for subdir, dirs, files in sorted(os.walk(dir_eeg_desikan)):
    
    for file in files:
        
        if 'conn_desi_cohi_time_eeg_theta' in (os.path.join(subdir,file)):
        
            thresholdAnalysis(s,'eeg_theta','dsk',os.path.join(subdir, file), threshold_values)
            s += 1


In [ ]:
# Desikan atlas - EEG theta band

componentAnalysis(1,'eeg','dsk', dir_eeg_desikan + '/sub01-7T/conn_desi_cohi_time_eeg_theta_subj1-7T_.mat', treshold_values)
#componentAnalysis(2,'eeg','dsk', dir_eeg_desikan + '/sub02-7T/conn_desi_cohi_time_eeg_theta_subj2-7T_.mat', treshold_values)
#componentAnalysis(3,'eeg','dsk', dir_eeg_desikan + '/sub03-7T/conn_desi_cohi_time_eeg_theta_subj3-7T_.mat', treshold_values)
#componentAnalysis(4,'eeg','dsk', dir_eeg_desikan + '/sub04-7T/conn_desi_cohi_time_eeg_theta_subj4-7T_.mat', treshold_values)
#componentAnalysis(5,'eeg','dsk', dir_eeg_desikan + '/sub05-7T/conn_desi_cohi_time_eeg_theta_subj5-7T_.mat', treshold_values)
#componentAnalysis(6,'eeg','dsk', dir_eeg_desikan + '/sub06-7T/conn_desi_cohi_time_eeg_theta_subj6-7T_.mat', treshold_values)
#componentAnalysis(7,'eeg','dsk', dir_eeg_desikan + '/sub07-7T/conn_desi_cohi_time_eeg_theta_subj7-7T_.mat', treshold_values)
#componentAnalysis(8,'eeg','dsk', dir_eeg_desikan + '/sub08-7T/conn_desi_cohi_time_eeg_theta_subj8-7T_.mat', treshold_values)
#componentAnalysis(9,'eeg','dsk', dir_eeg_desikan + '/sub09-7T/conn_desi_cohi_time_eeg_theta_subj9-7T_.mat', treshold_values)


In [36]:
# Destrieux atlas - EEG broad band

s = 1

for subdir, dirs, files in sorted(os.walk(dir_eeg_destrieux)):
    
    for file in files:
        
        if 'conn_destr_cohi_time_eeg_broad' in (os.path.join(subdir,file)):
        
            thresholdAnalysis(s,'eeg','dstrx',os.path.join(subdir, file), threshold_values)
            s += 1


conn_destr_cohi_time_eeg_theta_subj1-7T_.mat
conn_destr_cohi_time_eeg_alpha_subj1-7T_.mat
conn_destr_cohi_time_eeg_delta_subj1-7T_.mat
conn_destr_cohi_time_eeg_gamma_subj1-7T_.mat
conn_destr_cohi_time_eeg_beta_subj1-7T_.mat
conn_destr_cohi_time_eeg_broad_subj1-7T_.mat
Threshold analysis for subject 1, with eeg and using dstrx
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_destrieux/subj01-7T/conn_destr_cohi_time_eeg_broad_subj1-7T_.mat
Proportion kept: 0.5
Number of edges of thresholded graph: 54
Average degree of thresholded graph:  0.7297297297297297
Number of nodes of the giant component: 8
Number of nodes of the second biggest component: 5
___________________________________________________________________________
Proportion kept: 0.8
Number of edges of thresholded graph: 87
Average degree of thresholded graph:  1.1756756756756757
Number of nodes of the giant component: 17
Number of nodes of the second biggest component: 15
_____________________________________

Number of edges of thresholded graph: 2720
Average degree of thresholded graph:  36.75675675675676
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
conn_destr_cohi_time_eeg_gamma_subj2-7T_.mat
conn_destr_cohi_time_eeg_beta_subj2-7T_.mat
conn_destr_cohi_time_eeg_alpha_subj2-7T_.mat
conn_destr_cohi_time_eeg_delta_subj3-7T_.mat
conn_destr_cohi_time_eeg_gamma_subj3-7T_.mat
conn_destr_cohi_time_eeg_theta_subj3-7T_.mat
conn_destr_cohi_time_eeg_alpha_subj3-7T_.mat
conn_des

Number of edges of thresholded graph: 2720
Average degree of thresholded graph:  36.75675675675676
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
conn_destr_cohi_time_eeg_delta_subj4-7T_.mat
conn_destr_cohi_time_eeg_theta_subj5-7T_.mat
conn_destr_cohi_time_eeg_broad_subj5-7T_.mat
Threshold analysis for subject 5, with eeg and using dstrx
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_destrieux/subj05-7T/conn_destr_cohi_time_eeg_broad_subj5-7T_

Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
conn_destr_cohi_time_eeg_alpha_subj6-7T_.mat
conn_destr_cohi_time_eeg_gamma_subj7-7T_.mat
conn_destr_cohi_time_eeg_beta_subj7-7T_.mat
conn_destr_cohi_time_eeg_broad_subj7-7T_.mat
Threshold analysis for subject 7, with eeg and using dstrx
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_destrieux/subj07-7T/conn_destr_cohi_time_eeg_broad_subj7-7T_.mat
Proportion kept: 0.5
Number of edges of thresholde

Number of edges of thresholded graph: 2720
Average degree of thresholded graph:  36.75675675675676
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
conn_destr_cohi_time_eeg_delta_subj8-7T_.mat
conn_destr_cohi_time_eeg_alpha_subj8-7T_.mat
conn_destr_cohi_time_eeg_beta_subj8-7T_.mat
conn_destr_cohi_time_eeg_gamma_subj8-7T_.mat
conn_destr_cohi_time_eeg_theta_subj8-7T_.mat
conn_destr_cohi_time_eeg_theta_subj9-7T_.mat
conn_destr_cohi_time_eeg_beta_subj9-7T_.mat
conn_dest

In [ ]:
# Destrieux atlas - EEG broad band

componentAnalysis(1,'eeg','dstrx', dir_eeg_destrieux + '/sub01-7T/conn_destr_cohi_time_eeg_broad_subj1-7T_.mat', treshold_values)
#componentAnalysis(2,'eeg','dstrx', dir_eeg_destrieux + '/sub02-7T/conn_destr_cohi_time_eeg_broad_subj2-7T_.mat', treshold_values)
#componentAnalysis(3,'eeg','dstrx', dir_eeg_destrieux + '/sub03-7T/conn_destr_cohi_time_eeg_broad_subj3-7T_.mat', treshold_values)
#componentAnalysis(4,'eeg','dstrx', dir_eeg_destrieux + '/sub04-7T/conn_destr_cohi_time_eeg_broad_subj4-7T_.mat', treshold_values)
#componentAnalysis(5,'eeg','dstrx', dir_eeg_destrieux + '/sub05-7T/conn_destr_cohi_time_eeg_broad_subj5-7T_.mat', treshold_values)
#componentAnalysis(6,'eeg','dstrx', dir_eeg_destrieux + '/sub06-7T/conn_destr_cohi_time_eeg_broad_subj6-7T_.mat', treshold_values)
#componentAnalysis(7,'eeg','dstrx', dir_eeg_destrieux + '/sub07-7T/conn_destr_cohi_time_eeg_broad_subj7-7T_.mat', treshold_values)
#componentAnalysis(8,'eeg','dstrx', dir_eeg_destrieux + '/sub08-7T/conn_destr_cohi_time_eeg_broad_subj8-7T_.mat', treshold_values)
#componentAnalysis(9,'eeg','dstrx', dir_eeg_destrieux + '/sub09-7T/conn_destr_cohi_time_eeg_broad_subj9-7T_.mat', treshold_values)

In [37]:
# Destrieux atlas - EEG alpha band

s = 1

for subdir, dirs, files in sorted(os.walk(dir_eeg_destrieux)):
    
    for file in files:
        
        if 'conn_destr_cohi_time_eeg_alpha' in (os.path.join(subdir,file)):
        
            thresholdAnalysis(s,'eeg_alpha','dstrx',os.path.join(subdir, file), threshold_values)
            s += 1


conn_destr_cohi_time_eeg_theta_subj1-7T_.mat
conn_destr_cohi_time_eeg_alpha_subj1-7T_.mat
Threshold analysis for subject 1, with eeg_alpha and using dstrx
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_destrieux/subj01-7T/conn_destr_cohi_time_eeg_alpha_subj1-7T_.mat
Proportion kept: 0.5
Number of edges of thresholded graph: 54
Average degree of thresholded graph:  0.7297297297297297
Number of nodes of the giant component: 17
Number of nodes of the second biggest component: 10
___________________________________________________________________________
Proportion kept: 0.8
Number of edges of thresholded graph: 87
Average degree of thresholded graph:  1.1756756756756757
Number of nodes of the giant component: 28
Number of nodes of the second biggest component: 20
___________________________________________________________________________
Proportion kept: 1.0
Number of edges of thresholded graph: 109
Average degree of thresholded graph:  1.472972972972973
Number of nod

Number of edges of thresholded graph: 2720
Average degree of thresholded graph:  36.75675675675676
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
conn_destr_cohi_time_eeg_delta_subj3-7T_.mat
conn_destr_cohi_time_eeg_gamma_subj3-7T_.mat
conn_destr_cohi_time_eeg_theta_subj3-7T_.mat
conn_destr_cohi_time_eeg_alpha_subj3-7T_.mat
Threshold analysis for subject 3, with eeg_alpha and using dstrx
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_destrieux

Number of edges of thresholded graph: 2720
Average degree of thresholded graph:  36.75675675675676
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
conn_destr_cohi_time_eeg_broad_subj4-7T_.mat
conn_destr_cohi_time_eeg_delta_subj4-7T_.mat
conn_destr_cohi_time_eeg_theta_subj5-7T_.mat
conn_destr_cohi_time_eeg_broad_subj5-7T_.mat
conn_destr_cohi_time_eeg_delta_subj5-7T_.mat
conn_destr_cohi_time_eeg_alpha_subj5-7T_.mat
Threshold analysis for subject 5, with eeg_alpha and

Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
conn_destr_cohi_time_eeg_gamma_subj7-7T_.mat
conn_destr_cohi_time_eeg_beta_subj7-7T_.mat
conn_destr_cohi_time_eeg_broad_subj7-7T_.mat
conn_destr_cohi_time_eeg_delta_subj7-7T_.mat
conn_destr_cohi_time_eeg_theta_subj7-7T_.mat
conn_destr_cohi_time_eeg_alpha_subj7-7T_.mat
Threshold analysis for subject 7, with eeg_alpha and using dstrx
/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_destrieux/subj07-7T

Number of edges of thresholded graph: 2720
Average degree of thresholded graph:  36.75675675675676
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 50.0
Number of edges of thresholded graph: 5439
Average degree of thresholded graph:  73.5
Number of nodes of the giant component: 148
___________________________________________________________________________
Proportion kept: 100.0
Number of edges of thresholded graph: 10878
Average degree of thresholded graph:  147.0
Number of nodes of the giant component: 148
___________________________________________________________________________
conn_destr_cohi_time_eeg_beta_subj8-7T_.mat
conn_destr_cohi_time_eeg_gamma_subj8-7T_.mat
conn_destr_cohi_time_eeg_theta_subj8-7T_.mat
conn_destr_cohi_time_eeg_theta_subj9-7T_.mat
conn_destr_cohi_time_eeg_beta_subj9-7T_.mat
conn_destr_cohi_time_eeg_delta_subj9-7T_.mat
conn_destr_cohi_time_eeg_gamma_subj9-7T_.mat
conn_dest

In [ ]:
# Destrieux atlas - EEG alpha band

componentAnalysis(1,'eeg','dstrx', dir_eeg_destrieux + '/sub01-7T/conn_destr_cohi_time_eeg_alpha_subj1-7T_.mat', treshold_values)
#componentAnalysis(2,'eeg','dstrx', dir_eeg_destrieux + '/sub02-7T/conn_destr_cohi_time_eeg_alpha_subj2-7T_.mat', treshold_values)
#componentAnalysis(3,'eeg','dstrx', dir_eeg_destrieux + '/sub03-7T/conn_destr_cohi_time_eeg_alpha_subj3-7T_.mat', treshold_values)
#componentAnalysis(4,'eeg','dstrx', dir_eeg_destrieux + '/sub04-7T/conn_destr_cohi_time_eeg_alpha_subj4-7T_.mat', treshold_values)
#componentAnalysis(5,'eeg','dstrx', dir_eeg_destrieux + '/sub05-7T/conn_destr_cohi_time_eeg_alpha_subj5-7T_.mat', treshold_values)
#componentAnalysis(6,'eeg','dstrx', dir_eeg_destrieux + '/sub06-7T/conn_destr_cohi_time_eeg_alpha_subj6-7T_.mat', treshold_values)
#componentAnalysis(7,'eeg','dstrx', dir_eeg_destrieux + '/sub07-7T/conn_destr_cohi_time_eeg_alpha_subj7-7T_.mat', treshold_values)
#componentAnalysis(8,'eeg','dstrx', dir_eeg_destrieux + '/sub08-7T/conn_destr_cohi_time_eeg_alpha_subj8-7T_.mat', treshold_values)
#componentAnalysis(9,'eeg','dstrx', dir_eeg_destrieux + '/sub09-7T/conn_destr_cohi_time_eeg_alpha_subj9-7T_.mat', treshold_values)


In [ ]:
# Destrieux atlas - EEG beta band

s = 1

for subdir, dirs, files in sorted(os.walk(dir_eeg_destrieux)):
    
    for file in files:
        
        if 'conn_destr_cohi_time_eeg_beta' in (os.path.join(subdir,file)):
        
            thresholdAnalysis(s,'eeg_beta','dstrx',os.path.join(subdir, file), threshold_values)
            s += 1


In [ ]:
# Destrieux atlas - EEG beta band

componentAnalysis(1,'eeg','dstrx', dir_eeg_destrieux + '/sub01-7T/conn_destr_cohi_time_eeg_beta_subj1-7T_.mat', treshold_values)
#componentAnalysis(2,'eeg','dstrx', dir_eeg_destrieux + '/sub02-7T/conn_destr_cohi_time_eeg_beta_subj2-7T_.mat', treshold_values)
#componentAnalysis(3,'eeg','dstrx', dir_eeg_destrieux + '/sub03-7T/conn_destr_cohi_time_eeg_beta_subj3-7T_.mat', treshold_values)
#componentAnalysis(4,'eeg','dstrx', dir_eeg_destrieux + '/sub04-7T/conn_destr_cohi_time_eeg_beta_subj4-7T_.mat', treshold_values)
#componentAnalysis(5,'eeg','dstrx', dir_eeg_destrieux + '/sub05-7T/conn_destr_cohi_time_eeg_beta_subj5-7T_.mat', treshold_values)
#componentAnalysis(6,'eeg','dstrx', dir_eeg_destrieux + '/sub06-7T/conn_destr_cohi_time_eeg_beta_subj6-7T_.mat', treshold_values)
#componentAnalysis(7,'eeg','dstrx', dir_eeg_destrieux + '/sub07-7T/conn_destr_cohi_time_eeg_beta_subj7-7T_.mat', treshold_values)
#componentAnalysis(8,'eeg','dstrx', dir_eeg_destrieux + '/sub08-7T/conn_destr_cohi_time_eeg_beta_subj8-7T_.mat', treshold_values)
#componentAnalysis(9,'eeg','dstrx', dir_eeg_destrieux + '/sub09-7T/conn_destr_cohi_time_eeg_beta_subj9-7T_.mat', treshold_values)

In [ ]:
# Destrieux atlas - EEG delta band

s = 1

for subdir, dirs, files in sorted(os.walk(dir_eeg_destrieux)):
    
    for file in files:
        
        if 'conn_destr_cohi_time_eeg_delta' in (os.path.join(subdir,file)):
        
            thresholdAnalysis(s,'eeg_delta','dstrx',os.path.join(subdir, file), threshold_values)
            s += 1


In [ ]:
# Destrieux atlas - EEG delta band

componentAnalysis(1,'eeg','dstrx', dir_eeg_destrieux + '/sub01-7T/conn_destr_cohi_time_eeg_delta_subj1-7T_.mat', treshold_values)
#componentAnalysis(2,'eeg','dstrx', dir_eeg_destrieux + '/sub02-7T/conn_destr_cohi_time_eeg_delta_subj2-7T_.mat', treshold_values)
#componentAnalysis(3,'eeg','dstrx', dir_eeg_destrieux + '/sub03-7T/conn_destr_cohi_time_eeg_delta_subj3-7T_.mat', treshold_values)
#componentAnalysis(4,'eeg','dstrx', dir_eeg_destrieux + '/sub04-7T/conn_destr_cohi_time_eeg_delta_subj4-7T_.mat', treshold_values)
#componentAnalysis(5,'eeg','dstrx', dir_eeg_destrieux + '/sub05-7T/conn_destr_cohi_time_eeg_delta_subj5-7T_.mat', treshold_values)
#componentAnalysis(6,'eeg','dstrx', dir_eeg_destrieux + '/sub06-7T/conn_destr_cohi_time_eeg_delta_subj6-7T_.mat', treshold_values)
#componentAnalysis(7,'eeg','dstrx', dir_eeg_destrieux + '/sub07-7T/conn_destr_cohi_time_eeg_delta_subj7-7T_.mat', treshold_values)
#componentAnalysis(8,'eeg','dstrx', dir_eeg_destrieux + '/sub08-7T/conn_destr_cohi_time_eeg_delta_subj8-7T_.mat', treshold_values)
#componentAnalysis(9,'eeg','dstrx', dir_eeg_destrieux + '/sub09-7T/conn_destr_cohi_time_eeg_delta_subj9-7T_.mat', treshold_values)

In [ ]:
# Destrieux atlas - EEG gamma band

s = 1

for subdir, dirs, files in sorted(os.walk(dir_eeg_destrieux)):
    
    for file in files:
        
        if 'conn_destr_cohi_time_eeg_gamma' in (os.path.join(subdir,file)):
        
            thresholdAnalysis(s,'eeg_gamma','dstrx',os.path.join(subdir, file), threshold_values)
            s += 1


In [ ]:
# Destrieux atlas - EEG gamma band

componentAnalysis(1,'eeg','dstrx', dir_eeg_destrieux + '/sub01-7T/conn_destr_cohi_time_eeg_gamma_subj1-7T_.mat', treshold_values)
#componentAnalysis(2,'eeg','dstrx', dir_eeg_destrieux + '/sub02-7T/conn_destr_cohi_time_eeg_gamma_subj2-7T_.mat', treshold_values)
#componentAnalysis(3,'eeg','dstrx', dir_eeg_destrieux + '/sub03-7T/conn_destr_cohi_time_eeg_gamma_subj3-7T_.mat', treshold_values)
#componentAnalysis(4,'eeg','dstrx', dir_eeg_destrieux + '/sub04-7T/conn_destr_cohi_time_eeg_gamma_subj4-7T_.mat', treshold_values)
#componentAnalysis(5,'eeg','dstrx', dir_eeg_destrieux + '/sub05-7T/conn_destr_cohi_time_eeg_gamma_subj5-7T_.mat', treshold_values)
#componentAnalysis(6,'eeg','dstrx', dir_eeg_destrieux + '/sub06-7T/conn_destr_cohi_time_eeg_gamma_subj6-7T_.mat', treshold_values)
#componentAnalysis(7,'eeg','dstrx', dir_eeg_destrieux + '/sub07-7T/conn_destr_cohi_time_eeg_gamma_subj7-7T_.mat', treshold_values)
#componentAnalysis(8,'eeg','dstrx', dir_eeg_destrieux + '/sub08-7T/conn_destr_cohi_time_eeg_gamma_subj8-7T_.mat', treshold_values)
#componentAnalysis(9,'eeg','dstrx', dir_eeg_destrieux + '/sub09-7T/conn_destr_cohi_time_eeg_gamma_subj9-7T_.mat', treshold_values)

In [ ]:
# Destrieux atlas - EEG theta band

s = 1

for subdir, dirs, files in sorted(os.walk(dir_eeg_destrieux)):
    
    for file in files:
        
        if 'conn_destr_cohi_time_eeg_theta' in (os.path.join(subdir,file)):
        
            thresholdAnalysis(s,'eeg_theta','dstrx',os.path.join(subdir, file), threshold_values)
            s += 1


In [ ]:
# Destrieux atlas - EEG theta band

componentAnalysis(1,'eeg','dstrx', dir_eeg_destrieux + '/sub01-7T/conn_destr_cohi_time_eeg_theta_subj1-7T_.mat', treshold_values)
#componentAnalysis(2,'eeg','dstrx', dir_eeg_destrieux + '/sub02-7T/conn_destr_cohi_time_eeg_theta_subj2-7T_.mat', treshold_values)
#componentAnalysis(3,'eeg','dstrx', dir_eeg_destrieux + '/sub03-7T/conn_destr_cohi_time_eeg_theta_subj3-7T_.mat', treshold_values)
#componentAnalysis(4,'eeg','dstrx', dir_eeg_destrieux + '/sub04-7T/conn_destr_cohi_time_eeg_theta_subj4-7T_.mat', treshold_values)
#componentAnalysis(5,'eeg','dstrx', dir_eeg_destrieux + '/sub05-7T/conn_destr_cohi_time_eeg_theta_subj5-7T_.mat', treshold_values)
#componentAnalysis(6,'eeg','dstrx', dir_eeg_destrieux + '/sub06-7T/conn_destr_cohi_time_eeg_theta_subj6-7T_.mat', treshold_values)
#componentAnalysis(7,'eeg','dstrx', dir_eeg_destrieux + '/sub07-7T/conn_destr_cohi_time_eeg_theta_subj7-7T_.mat', treshold_values)
#componentAnalysis(8,'eeg','dstrx', dir_eeg_destrieux + '/sub08-7T/conn_destr_cohi_time_eeg_theta_subj8-7T_.mat', treshold_values)
#componentAnalysis(9,'eeg','dstrx', dir_eeg_destrieux + '/sub09-7T/conn_destr_cohi_time_eeg_theta_subj9-7T_.mat', treshold_values)

In [ ]:
#TO TEST IF NO MEMORY PROBLEMS - TODO TODO TODO
for subdir, dirs, files in sorted(os.walk(dir_fmri_desikan)):
    
    for file in files:
        #print(os.path.join(subdir, file))